# 03 — Which saved artifact do we serve?

`src/train.py --full-refit` writes two strategies into one bundle:

| strategy | what it is |
|---|---|
| `fold_average` | the five CV fold models, averaged — 5 members at weight 0.2 |
| `full_refit` | one model fit on 100% of train at a fixed round count — 1 member at 1.0 |

**This notebook does not train anything.** Both runs went through `src.train.run`
so they are in `experiments.csv` with a fold fingerprint. The conclusions live in
`experiments.md`; this file is how the numbers are reproduced.

## The methodological problem, first

**Strategy (b) has no CV score.** A model fit on 100% of train has no
out-of-fold rows, so there is no honest number for *that artifact*. There is
one for its *procedure* — committing to a fixed round count instead of watching
a validation fold — and that is run `1024_143040_lgbm`.

And the leaderboard cannot settle it either: `experiments.md` → "What the public
LB can and cannot resolve" established paired LB noise of ~0.00008 against
changes of 0.00010–0.00018. So **no submission is spent here.** Three free
measurements instead.

In [ ]:
import sys

sys.path.insert(0, "..")


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from src import config as C
from src import cv as CV
from src import predict as P
from src.utils import savefig

log = pd.read_csv(C.ROOT / "experiments.csv").set_index("run_id")
BUNDLE = [r for r in log.index if "repackage" in str(log.loc[r, "notes"])][-1]
FIXED = [r for r in log.index if "fixed 1044 rounds" in str(log.loc[r, "notes"])][-1]
REF = "0904_202729_lgbm"
BUNDLE, FIXED

## 1. What does the procedure cost? (the only legitimate CV number for (b))

Strategy (b) cannot early-stop, because a 100%-of-train fit has nothing held
out. So it has to carry the round count over from CV. Run `FIXED` prices exactly
that: same folds, same params, `--rounds 1044 --no-early-stopping`.

1044 = `round(mean(best_iterations) × 5/4)` — the fold models stopped at
712/937/833/822/873, and a refit sees 1/(k−1) more data than a fold did.

In [ ]:
y = (pd.read_csv(C.TRAIN_FILE, usecols=["Churn"])["Churn"] == "Yes").astype("int8").to_numpy()
folds = CV.get_folds(pd.DataFrame({C.TARGET: y}))

oof_es = np.load(C.EXPERIMENTS / f"oof_{REF}.npy")
oof_fixed = np.load(C.EXPERIMENTS / f"oof_{FIXED}.npy")

deltas = [CV.score(y[va], oof_fixed[va]) - CV.score(y[va], oof_es[va]) for _, va in folds]
print(f"early stopping : {log.cv_score[REF]:.6f}")
print(f"fixed 1044     : {log.cv_score[FIXED]:.6f}")
print(f"delta          : {log.cv_score[FIXED] - log.cv_score[REF]:+.6f}")
print(f"per fold       : {', '.join(f'{d:+.6f}' for d in deltas)}")
print(f"same sign in   : {sum(d < 0 for d in deltas)}/5 folds (negative)")
print("seed spread from experiments.md: +/-0.000052")

**Read.** −0.000034, negative in **5/5 folds**. By this repo's own standard
(`experiments.md` → Consequences) the *sign* is the believable part: 5/5
same-sign is real, while the magnitude sits below the ±0.000052 seed spread.

So dropping early stopping costs a little, consistently. Not enough to matter on
its own — but it is a cost with no offsetting benefit.

## 2. Do the two artifacts actually disagree?

No training needed: score the real test set through both strategies of the same
bundle. AUC is a ranking metric and the business use is a ranked retention call
list (`notebooks/99_final_writeup.ipynb`), so the question is not "do the
probabilities differ" but **"does the call list differ"**.

In [ ]:
raw_test = pd.read_csv(C.TEST_FILE)
p_fold = P.predict_raw(P.load_bundle(BUNDLE, "fold_average"), raw_test)
p_refit = P.predict_raw(P.load_bundle(BUNDLE, "full_refit"), raw_test)

print(f"spearman        : {spearmanr(p_fold, p_refit).statistic:.8f}")
print(f"mean |delta p|  : {np.abs(p_fold - p_refit).mean():.6f}")
print(f"max  |delta p|  : {np.abs(p_fold - p_refit).max():.6f}")

d_fold = pd.qcut(p_fold, 10, labels=False)
d_refit = pd.qcut(p_refit, 10, labels=False)
print(f"changed decile  : {(d_fold != d_refit).mean() * 100:.2f}%")
print(f"moved >1 decile : {(np.abs(d_fold - d_refit) > 1).mean() * 100:.3f}%")

depths = np.array([0.01, 0.05, 0.10, 0.20, 0.30, 0.50])
overlap = []
for depth in depths:
    k = int(len(p_fold) * depth)
    top_a = set(np.argsort(-p_fold)[:k])
    top_b = set(np.argsort(-p_refit)[:k])
    overlap.append(len(top_a & top_b) / k)
    print(f"top {depth * 100:>4.0f}% overlap : {overlap[-1] * 100:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

band = pd.cut(p_fold, [0, 0.1, 0.3, 0.5, 0.7, 1.0], include_lowest=True)
by_band = (
    pd.DataFrame({"band": band, "d": np.abs(p_fold - p_refit)})
    .groupby("band", observed=True)["d"]
    .agg(["mean", "count"])
)
ax1.bar(range(len(by_band)), by_band["mean"], color="#4C72B0")
ax1.set_xticks(range(len(by_band)))
ax1.set_xticklabels([str(i) for i in by_band.index], rotation=20, ha="right", fontsize=8)
ax1.set_xlabel("fold_average predicted probability")
ax1.set_ylabel("mean |delta p|")
ax1.set_title("Where the two strategies disagree")
for i, (m, n) in enumerate(zip(by_band["mean"], by_band["count"])):
    ax1.text(i, m, f"{n / len(p_fold) * 100:.0f}% of rows", ha="center", va="bottom", fontsize=7)

ax2.plot(depths * 100, np.array(overlap) * 100, "o-", color="#C44E52")
ax2.axhline(100, ls=":", c="grey", lw=1)
ax2.set_xlabel("call-list depth (% of customers contacted)")
ax2.set_ylabel("overlap between the two lists (%)")
ax2.set_title("The lists a retention team would actually work")
ax2.set_ylim(90, 100.5)

fig.suptitle("fold_average vs full_refit, 254,655 test rows", y=1.02)
savefig(fig, "03_serving_strategy")

**Read.** The disagreement is real but small, and it is smallest exactly where
the model is confident — 55% of customers sit below p=0.1 with a mean
disagreement of 0.002. 9% of customers change decile, but almost none moves more
than one. The top 10% of the list, which is where the writeup concluded the
retention budget should go, is 97% the same people.

Operationally, then, the two strategies are near-interchangeable. Which means
the choice has to be made on something other than score.

## 3. The verdict, and why it is not about score

| | `fold_average` | `full_refit` |
|---|---|---|
| OOF for the artifact | 0.916391 (conservative — see below) | **none exists** |
| procedure cost on CV | — | −0.000034, 5/5 folds |
| verifiable against the LB | **yes, bitwise** | no |
| bundle size | 30MB | 6MB |
| members to score | 5 | 1 |

**Serve `fold_average`.** Not because it scores better — (b)'s artifact has no
score to compare. Because:

1. It is **verifiable**. `test_*.npy` is the mean of the five fold models, and
   `tests/test_predict.py` asserts that scoring `data/raw/test.csv` through this
   strategy reproduces that array bitwise. So the served artifact is provably the
   one behind public LB 0.91393. Nothing equivalent can be said of a refit.
2. Its procedure costs nothing, while (b)'s costs a small, consistently-signed
   amount.
3. The operational difference is ~3% of the top-decile call list, which is not
   worth buying unfalsifiability with.

Note the framing, because it is easy to get backwards: OOF 0.916391 is measured
on **one fold model per row**, so it is a *conservative estimate* of the
five-model average, not a description of it.

`full_refit` stays in the bundle. It is one env var away (`MODEL_STRATEGY`) if a
future constraint — cold-start latency, image size — ever makes 6MB and one
member worth the loss of verifiability.